# 01 — Load Tier 1 data

Read raw CSV/XLSX files. Print shapes and dtypes. Cache typed copies as parquet so downstream notebooks do not pay Excel-parse cost.

**Upstream:** raw files in `data/`

**Output:** parquets in `pipeline/artifacts/` (sales, cb, tpr, inv_snapshot, item_master, vendor, po, slprsn_key)

**Promotes to:** `src/load.py` once verified.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'pipeline' else Path.cwd()
DATA = ROOT / 'data'
ART  = ROOT / 'pipeline' / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)

SALES_CSV   = DATA / 'POP_SalesTransactionHistory.csv'
INV_XLSX    = DATA / 'POP_InventorySnapshot.xlsx'
ITEM_XLSX   = DATA / 'POP_ItemSpecMaster.xlsx'
VENDOR_XLSX = DATA / 'POP_VendorMaster.xlsx'
PO_XLSX     = DATA / 'POP_PurchaseOrderHistory.XLSX'
CB_XLSX     = DATA / 'POP_ChargeBack_Deductions_Penalties_Freight.xlsx'
SLPRSN_XLSX = DATA / 'SLPRSNID_SALESCHANNEL_KEY.xlsx'

for p in [SALES_CSV, INV_XLSX, ITEM_XLSX, VENDOR_XLSX, PO_XLSX, CB_XLSX, SLPRSN_XLSX]:
    assert p.exists(), f'missing: {p}'

## 2. Read raw files

One source-of-truth load per file. Downstream notebooks read the cached parquets in step 5.

In [ ]:
sales = pd.read_csv(SALES_CSV, parse_dates=['DOCDATE'], low_memory=False)

inv = pd.concat([
    pd.read_excel(INV_XLSX, sheet_name='Site 1 - SF').assign(DC='SF'),
    pd.read_excel(INV_XLSX, sheet_name='Site 2 - NJ').assign(DC='NJ'),
    pd.read_excel(INV_XLSX, sheet_name='Site 3 - LA').assign(DC='LA'),
], ignore_index=True)

items   = pd.read_excel(ITEM_XLSX,   sheet_name='Item Spec Master')
vendors = pd.read_excel(VENDOR_XLSX, sheet_name='Supplier Master')
pos     = pd.read_excel(PO_XLSX,     sheet_name='PO Order History 2023-2025')
cb      = pd.read_excel(CB_XLSX,     sheet_name='Data - Deductions & Cause Code')
slprsn  = pd.read_excel(SLPRSN_XLSX)   # default first sheet; adjust if needed

for name, df in [('sales',sales),('inv',inv),('items',items),('vendors',vendors),('pos',pos),('cb',cb),('slprsn',slprsn)]:
    print(f'{name:10s} {df.shape}  cols[:6]={list(df.columns)[:6]}')

## 3. Derive TPR subset of chargebacks

Chargebacks have many cause codes. The TPR / promo codes are what drive `is_promo`; split them out here so downstream notebooks can grab just `tpr.parquet`.

In [ ]:
tpr_mask = cb['Cause Code Desc'].str.contains('TPR|promo|price reduction', case=False, na=False)
tpr = cb[tpr_mask].copy()

print(f'tpr rows: {len(tpr)} of {len(cb)} chargeback rows ({len(tpr)/len(cb):.1%})')
print()
print('Top cause codes in tpr subset:')
print(tpr['Cause Code'].value_counts().head(15))

## 4. Validate

In [ ]:
print('sales date range   :', sales['DOCDATE'].min(), '->', sales['DOCDATE'].max())
print('unique SKUs (sales):', sales['ITEMNMBR'].nunique())
print('unique SKUs (items):', items.iloc[:, 0].nunique())
print('inv rows per DC    :', inv['DC'].value_counts().to_dict())
print('tpr customers      :', tpr['Customer Number'].nunique() if 'Customer Number' in tpr.columns else 'col missing')
print()
print('sales columns with nulls (top 10):')
print(sales.isna().sum().sort_values(ascending=False).head(10))

## 5. Save downstream artifact

In [ ]:
to_cache = {
    'sales':         sales,
    'cb':            cb,
    'tpr':           tpr,
    'inv_snapshot':  inv,
    'item_master':   items,
    'vendor_master': vendors,
    'po':            pos,
    'slprsn_key':    slprsn,
}

for name, df in to_cache.items():
    path = ART / f'{name}.parquet'
    try:
        df.to_parquet(path)
        print(f'{name:14s} {df.shape} -> {path.name}')
    except Exception as e:
        alt = ART / f'{name}.pkl'
        df.to_pickle(alt)
        print(f'{name:14s} parquet failed ({type(e).__name__}); saved pickle -> {alt.name}')

## 6. Promote

Once loads and sanity checks look right, move the read logic into `src/load.py` as a single `load_all()` returning a dict of dataframes. Downstream dev notebooks can then:

```python
from src.load import load_all
data = load_all()
sales, cb, tpr = data['sales'], data['cb'], data['tpr']
```

Keep this notebook as the human-readable "how the data was parsed" reference.